# 03 — Urban Attractiveness Index and spatial analysis

## Purpose

This notebook reproduces the final analytical stage reported in the manuscript:

1. merge building-level accessibility and population aged 65 years and over;
2. calculate functional diversity;
3. assess PCA suitability using KMO and Bartlett's test;
4. derive PCA weights and construct the Urban Attractiveness Index;
5. compare PCA-derived and equal-weight indices;
6. calculate the Ageing–Accessibility Vulnerability Index (IVEA);
7. calculate Global Moran's I for 8, 12 and 16 nearest neighbours;
8. calculate LISA clusters using 16 nearest neighbours;
9. aggregate the index to BGRI subsections and 100 m, 250 m and 500 m grids;
10. reproduce the spatial-aggregation results reported in Table 3;
11. export the final analytical datasets.

## Inputs

Place these files in the repository `data/` directory:

- `merged_results.csv`
- `population_65plus.csv`
- `BGRI2021_1312.gpkg`

## Scope

The notebook excludes exploratory Folium maps, duplicate map versions, widgets, zoom tests,
temporary exports and visual checks. It retains only the calculations required by the article.

The walking-speed sensitivity analysis is not included here because it was not implemented
as a complete reproducible block in the supplied PCA notebook.

In [ ]:
from pathlib import Path

import geopandas as gpd
import libpysal
import numpy as np
import pandas as pd

from esda.moran import Moran, Moran_Local
from factor_analyzer.factor_analyzer import (
    calculate_bartlett_sphericity,
    calculate_kmo,
)
from scipy.stats import pearsonr
from shapely import wkt
from shapely.geometry import box
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve().parent

DATA_DIR = PROJECT_ROOT / "data" / "final"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
SPATIAL_DIR = PROJECT_ROOT / "outputs" / "spatial"

ACCESSIBILITY_FILE = DATA_DIR / "merged_results.csv"
POPULATION_FILE = DATA_DIR / "population_65plus.csv"
BGRI_FILE = DATA_DIR / "BGRI2021_1312.gpkg"

BUILDING_OUTPUT = SPATIAL_DIR / "building_level_results.gpkg"
BUILDING_CSV_OUTPUT = TABLE_DIR / "building_level_results.csv"
PCA_WEIGHTS_OUTPUT = TABLE_DIR / "pca_weights.csv"
PCA_DIAGNOSTICS_OUTPUT = TABLE_DIR / "pca_diagnostics.csv"
MORAN_SENSITIVITY_OUTPUT = TABLE_DIR / "moran_knn_sensitivity.csv"
LISA_COUNTS_OUTPUT = TABLE_DIR / "lisa_cluster_counts.csv"
TABLE3_OUTPUT = TABLE_DIR / "table3_spatial_aggregation_knn16.csv"

METRIC_CRS = "EPSG:3763"
BGRI_ID_COLUMN = "DTMNFRSEC21"

K_VALUES = [8, 12, 16]
FINAL_K = 16
PERMUTATIONS = 999
RANDOM_SEED = 42
GRID_SIZES_M = [100, 250, 500]

TABLE_DIR.mkdir(parents=True, exist_ok=True)
SPATIAL_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ---------------------------------------------------------------------
# Load and merge building accessibility and population
# ---------------------------------------------------------------------

accessibility = pd.read_csv(ACCESSIBILITY_FILE, low_memory=False)
population = pd.read_csv(POPULATION_FILE, low_memory=False)

required_accessibility_columns = {
    "osm_id",
    "geometry_wkt",
    "tempo_minimo_servico_seg",
    "Supermercados",
    "Bancos",
    "Farmacias",
    "CTT",
    "Parques e jardins",
    "Centro Saude",
    "Hospitais",
}
missing_accessibility = sorted(
    required_accessibility_columns.difference(accessibility.columns)
)
if missing_accessibility:
    raise KeyError(
        "Missing columns in merged_results.csv: "
        + ", ".join(missing_accessibility)
    )

if not {"osm_id", "pop_64_mais"}.issubset(population.columns):
    raise KeyError(
        "population_65plus.csv must contain 'osm_id' and 'population_65plus.csv'."
    )

data = accessibility.merge(
    population[["osm_id", "pop_64_mais"]],
    on="osm_id",
    how="left",
    validate="one_to_one",
)

data["pop_64_mais"] = pd.to_numeric(
    data["pop_64_mais"],
    errors="coerce",
).fillna(0)

data["geometry"] = data["geometry_wkt"].apply(wkt.loads)
gdf = gpd.GeoDataFrame(
    data,
    geometry="geometry",
    crs=METRIC_CRS,
)

if gdf["osm_id"].duplicated().any():
    raise ValueError("Duplicated osm_id values were found after the merge.")

print(f"Buildings loaded: {len(gdf):,}")
print(f"Allocated population aged 65+: {gdf['pop_64_mais'].sum():,.0f}")

In [ ]:
# ---------------------------------------------------------------------
# Functional diversity and index variables
# ---------------------------------------------------------------------

SERVICE_COLUMNS = [
    "Centro Saude",
    "Farmacias",
    "Hospitais",
    "Supermercados",
    "Bancos",
    "CTT",
    "Parques e jardins",
]

for column in SERVICE_COLUMNS + ["tempo_minimo_servico_seg"]:
    gdf[column] = pd.to_numeric(gdf[column], errors="coerce")

gdf["diversidade_servicos"] = (
    gdf[SERVICE_COLUMNS]
    .fillna(0)
    .gt(0)
    .sum(axis=1)
)

index_variables = pd.DataFrame(
    {
        "supermarkets": gdf["Supermercados"].fillna(0),
        "banks": gdf["Bancos"].fillna(0),
        "pharmacies": gdf["Farmacias"].fillna(0),
        "post_offices": gdf["CTT"].fillna(0),
        "parks_and_gardens": gdf["Parques e jardins"].fillna(0),
        "health_centres": gdf["Centro Saude"].fillna(0),
        "hospitals": gdf["Hospitais"].fillna(0),
        "pedestrian_proximity": 1 / (
            gdf["tempo_minimo_servico_seg"].fillna(0) + 1
        ),
    },
    index=gdf.index,
)

In [ ]:
# ---------------------------------------------------------------------
# Min–max normalisation
# ---------------------------------------------------------------------

scaler = MinMaxScaler()
normalised_variables = pd.DataFrame(
    scaler.fit_transform(index_variables),
    columns=index_variables.columns,
    index=index_variables.index,
)

In [ ]:
# ---------------------------------------------------------------------
# PCA suitability: KMO and Bartlett's test
# ---------------------------------------------------------------------

kmo_per_variable, kmo_overall = calculate_kmo(
    normalised_variables
)
bartlett_chi_square, bartlett_p_value = (
    calculate_bartlett_sphericity(normalised_variables)
)

print(f"Overall KMO: {kmo_overall:.3f}")
print(
    "Bartlett's test: "
    f"chi-square={bartlett_chi_square:,.3f}, "
    f"p={bartlett_p_value:.6g}"
)

In [ ]:
# ---------------------------------------------------------------------
# PCA weights and Urban Attractiveness Index
# ---------------------------------------------------------------------

pca = PCA(n_components=1, random_state=RANDOM_SEED)
pca.fit(normalised_variables)

absolute_loadings = np.abs(pca.components_[0])
pca_weights = absolute_loadings / absolute_loadings.sum()
explained_variance_pc1 = float(
    pca.explained_variance_ratio_[0]
)

weights_table = pd.DataFrame(
    {
        "variable": normalised_variables.columns,
        "absolute_loading_pc1": absolute_loadings,
        "pca_weight": pca_weights,
        "kmo_variable": kmo_per_variable,
    }
).sort_values("pca_weight", ascending=False)

gdf["urban_attractiveness_index"] = np.dot(
    normalised_variables.values,
    pca_weights,
)
gdf["urban_attractiveness_index_normalised"] = (
    MinMaxScaler().fit_transform(
        gdf[["urban_attractiveness_index"]]
    )
)

weights_table

In [ ]:
# ---------------------------------------------------------------------
# Equal-weight robustness analysis
# ---------------------------------------------------------------------

equal_weights = np.repeat(
    1 / normalised_variables.shape[1],
    normalised_variables.shape[1],
)

gdf["equal_weight_index"] = np.dot(
    normalised_variables.values,
    equal_weights,
)
gdf["equal_weight_index_normalised"] = (
    MinMaxScaler().fit_transform(
        gdf[["equal_weight_index"]]
    )
)

equal_weight_correlation, equal_weight_p = pearsonr(
    gdf["urban_attractiveness_index_normalised"],
    gdf["equal_weight_index_normalised"],
)

print(
    "PCA-derived versus equal-weight index: "
    f"r={equal_weight_correlation:.4f}, "
    f"p={equal_weight_p:.6g}"
)

In [ ]:
# ---------------------------------------------------------------------
# Ageing–Accessibility Vulnerability Index (IVEA)
# ---------------------------------------------------------------------

gdf["ivea"] = (
    gdf["pop_64_mais"]
    * (
        1
        - gdf["urban_attractiveness_index_normalised"]
    )
)

population_index_correlation, population_index_p = pearsonr(
    gdf["urban_attractiveness_index_normalised"],
    gdf["pop_64_mais"],
)

print(
    "Urban Attractiveness Index versus population aged 65+: "
    f"r={population_index_correlation:.4f}, "
    f"p={population_index_p:.6g}"
)

In [ ]:
# ---------------------------------------------------------------------
# Global Moran's I sensitivity analysis
# ---------------------------------------------------------------------

spatial_points = gdf.copy()
spatial_points["geometry"] = spatial_points.geometry.centroid
spatial_points = spatial_points.reset_index(drop=True)

moran_records = []

for k in K_VALUES:
    weights = libpysal.weights.KNN.from_dataframe(
        spatial_points,
        k=k,
    )
    weights.transform = "r"

    moran = Moran(
        spatial_points[
            "urban_attractiveness_index_normalised"
        ].fillna(0).values,
        weights,
        permutations=PERMUTATIONS,
    )

    moran_records.append(
        {
            "k": k,
            "global_morans_i": moran.I,
            "pseudo_p": moran.p_sim,
            "connected_components": weights.n_components,
        }
    )

moran_sensitivity = pd.DataFrame(moran_records)
moran_sensitivity

In [ ]:
# ---------------------------------------------------------------------
# LISA for the Urban Attractiveness Index
# ---------------------------------------------------------------------

lisa_weights = libpysal.weights.KNN.from_dataframe(
    spatial_points,
    k=FINAL_K,
)
lisa_weights.transform = "r"

lisa = Moran_Local(
    spatial_points[
        "urban_attractiveness_index_normalised"
    ].fillna(0).values,
    lisa_weights,
    permutations=PERMUTATIONS,
    seed=RANDOM_SEED,
)

spatial_points["lisa_quadrant"] = lisa.q
spatial_points["lisa_p"] = lisa.p_sim
spatial_points["lisa_significant"] = (
    spatial_points["lisa_p"] < 0.05
)
spatial_points["lisa_cluster"] = "Not significant"

cluster_labels = {
    1: "High–High",
    2: "Low–High",
    3: "Low–Low",
    4: "High–Low",
}

for quadrant, label in cluster_labels.items():
    mask = (
        spatial_points["lisa_significant"]
        & spatial_points["lisa_quadrant"].eq(quadrant)
    )
    spatial_points.loc[mask, "lisa_cluster"] = label

gdf["lisa_cluster"] = spatial_points[
    "lisa_cluster"
].values
gdf["lisa_p"] = spatial_points["lisa_p"].values

lisa_counts = (
    gdf["lisa_cluster"]
    .value_counts()
    .rename_axis("cluster")
    .reset_index(name="building_count")
)
lisa_counts

In [ ]:
# ---------------------------------------------------------------------
# LISA for the IVEA
# ---------------------------------------------------------------------

ivea_lisa = Moran_Local(
    gdf["ivea"].fillna(0).values,
    lisa_weights,
    permutations=PERMUTATIONS,
    seed=RANDOM_SEED,
)

gdf["ivea_lisa_quadrant"] = ivea_lisa.q
gdf["ivea_lisa_p"] = ivea_lisa.p_sim
gdf["ivea_lisa_significant"] = gdf[
    "ivea_lisa_p"
] < 0.05
gdf["ivea_lisa_cluster"] = "Not significant"

for quadrant, label in cluster_labels.items():
    mask = (
        gdf["ivea_lisa_significant"]
        & gdf["ivea_lisa_quadrant"].eq(quadrant)
    )
    gdf.loc[mask, "ivea_lisa_cluster"] = label

In [ ]:
# ---------------------------------------------------------------------
# Spatial aggregation helpers
# ---------------------------------------------------------------------

def create_regular_grid(
    source: gpd.GeoDataFrame,
    cell_size: int,
) -> gpd.GeoDataFrame:
    min_x, min_y, max_x, max_y = source.total_bounds

    cells = []
    cell_ids = []
    cell_id = 0

    x_value = min_x
    while x_value < max_x:
        y_value = min_y
        while y_value < max_y:
            cells.append(
                box(
                    x_value,
                    y_value,
                    x_value + cell_size,
                    y_value + cell_size,
                )
            )
            cell_ids.append(cell_id)
            cell_id += 1
            y_value += cell_size
        x_value += cell_size

    grid = gpd.GeoDataFrame(
        {"spatial_unit_id": cell_ids},
        geometry=cells,
        crs=source.crs,
    )

    study_area = source.geometry.union_all()
    return grid[grid.intersects(study_area)].copy()


def aggregate_index(
    buildings: gpd.GeoDataFrame,
    zones: gpd.GeoDataFrame,
    zone_id: str,
) -> gpd.GeoDataFrame:
    building_points = buildings[
        [
            "urban_attractiveness_index_normalised",
            "geometry",
        ]
    ].copy()
    building_points["geometry"] = (
        building_points.geometry.centroid
    )

    joined = gpd.sjoin(
        building_points,
        zones[[zone_id, "geometry"]],
        how="left",
        predicate="within",
    )

    statistics = (
        joined.groupby(zone_id)[
            "urban_attractiveness_index_normalised"
        ]
        .agg(["mean", "min", "max", "count"])
        .reset_index()
        .rename(
            columns={
                "mean": "index_mean",
                "min": "index_min",
                "max": "index_max",
                "count": "building_count",
            }
        )
    )
    statistics["within_unit_range"] = (
        statistics["index_max"]
        - statistics["index_min"]
    )

    result = zones.merge(
        statistics,
        on=zone_id,
        how="left",
    )
    return result.dropna(subset=["index_mean"]).copy()


def calculate_global_moran(
    zones: gpd.GeoDataFrame,
    value_column: str,
    k: int = FINAL_K,
) -> tuple[float, float]:
    points = zones.copy()
    points["geometry"] = points.geometry.centroid
    points = points.reset_index(drop=True)

    effective_k = min(k, len(points) - 1)
    if effective_k < 1:
        raise ValueError(
            "At least two spatial units are required."
        )

    weights = libpysal.weights.KNN.from_dataframe(
        points,
        k=effective_k,
    )
    weights.transform = "r"

    moran = Moran(
        points[value_column].fillna(0).values,
        weights,
        permutations=PERMUTATIONS,
    )
    return float(moran.I), float(moran.p_sim)

In [ ]:
# ---------------------------------------------------------------------
# Building-level reference values
# ---------------------------------------------------------------------

building_variance = gdf[
    "urban_attractiveness_index_normalised"
].var()

building_moran_row = moran_sensitivity.loc[
    moran_sensitivity["k"].eq(FINAL_K)
].iloc[0]

aggregation_records = [
    {
        "spatial_unit": "Building",
        "variance": building_variance,
        "variance_reduction_percent": 0.0,
        "global_morans_i": building_moran_row[
            "global_morans_i"
        ],
        "pseudo_p": building_moran_row["pseudo_p"],
        "mean_within_unit_range": 0.0,
        "maximum_within_unit_range": 0.0,
    }
]

In [ ]:
# ---------------------------------------------------------------------
# Regular-grid aggregation: 100 m, 250 m and 500 m
# ---------------------------------------------------------------------

for cell_size in GRID_SIZES_M:
    grid = create_regular_grid(gdf, cell_size)
    aggregated_grid = aggregate_index(
        gdf,
        grid,
        "spatial_unit_id",
    )

    variance = aggregated_grid["index_mean"].var()
    variance_reduction = (
        (building_variance - variance)
        / building_variance
        * 100
    )
    morans_i, pseudo_p = calculate_global_moran(
        aggregated_grid,
        "index_mean",
    )

    aggregation_records.append(
        {
            "spatial_unit": f"{cell_size} m grid",
            "variance": variance,
            "variance_reduction_percent": variance_reduction,
            "global_morans_i": morans_i,
            "pseudo_p": pseudo_p,
            "mean_within_unit_range": aggregated_grid[
                "within_unit_range"
            ].mean(),
            "maximum_within_unit_range": aggregated_grid[
                "within_unit_range"
            ].max(),
        }
    )

    aggregated_grid.to_file(
        SPATIAL_DIR / f"grid_{cell_size}m.gpkg",
        layer=f"grid_{cell_size}m",
        driver="GPKG",
    )

In [ ]:
# ---------------------------------------------------------------------
# BGRI statistical-subsection aggregation
# ---------------------------------------------------------------------

bgri = gpd.read_file(BGRI_FILE).to_crs(METRIC_CRS)

if BGRI_ID_COLUMN not in bgri.columns:
    raise KeyError(
        f"Missing BGRI identifier column: {BGRI_ID_COLUMN}"
    )

bgri_aggregated = aggregate_index(
    gdf,
    bgri,
    BGRI_ID_COLUMN,
)

bgri_variance = bgri_aggregated["index_mean"].var()
bgri_variance_reduction = (
    (building_variance - bgri_variance)
    / building_variance
    * 100
)
bgri_morans_i, bgri_pseudo_p = calculate_global_moran(
    bgri_aggregated,
    "index_mean",
)

aggregation_records.append(
    {
        "spatial_unit": "BGRI statistical subsections",
        "variance": bgri_variance,
        "variance_reduction_percent": bgri_variance_reduction,
        "global_morans_i": bgri_morans_i,
        "pseudo_p": bgri_pseudo_p,
        "mean_within_unit_range": bgri_aggregated[
            "within_unit_range"
        ].mean(),
        "maximum_within_unit_range": bgri_aggregated[
            "within_unit_range"
        ].max(),
    }
)

bgri_aggregated.to_file(
    SPATIAL_DIR / "bgri_aggregated.gpkg",
    layer="bgri_aggregated",
    driver="GPKG",
)

In [ ]:
# ---------------------------------------------------------------------
# Table 3
# ---------------------------------------------------------------------

table3 = pd.DataFrame(aggregation_records)

display_order = [
    "Building",
    "100 m grid",
    "BGRI statistical subsections",
    "250 m grid",
    "500 m grid",
]

table3["display_order"] = pd.Categorical(
    table3["spatial_unit"],
    categories=display_order,
    ordered=True,
)
table3 = (
    table3.sort_values("display_order")
    .drop(columns="display_order")
    .reset_index(drop=True)
)

table3_display = table3[
    [
        "spatial_unit",
        "variance",
        "variance_reduction_percent",
        "global_morans_i",
        "pseudo_p",
    ]
].copy()

table3_display.round(
    {
        "variance": 4,
        "variance_reduction_percent": 2,
        "global_morans_i": 3,
        "pseudo_p": 3,
    }
)

In [ ]:
# ---------------------------------------------------------------------
# Export analytical outputs
# ---------------------------------------------------------------------

weights_table.to_csv(
    PCA_WEIGHTS_OUTPUT,
    index=False,
)

pca_diagnostics = pd.DataFrame(
    [
        {
            "kmo_overall": kmo_overall,
            "bartlett_chi_square": bartlett_chi_square,
            "bartlett_p_value": bartlett_p_value,
            "explained_variance_pc1": explained_variance_pc1,
            "pca_equal_weight_correlation": equal_weight_correlation,
            "pca_equal_weight_p_value": equal_weight_p,
            "population_index_correlation": population_index_correlation,
            "population_index_p_value": population_index_p,
        }
    ]
)
pca_diagnostics.to_csv(
    PCA_DIAGNOSTICS_OUTPUT,
    index=False,
)

moran_sensitivity.to_csv(
    MORAN_SENSITIVITY_OUTPUT,
    index=False,
)
lisa_counts.to_csv(
    LISA_COUNTS_OUTPUT,
    index=False,
)
table3.to_csv(
    TABLE3_OUTPUT,
    index=False,
)

gdf["geometry_wkt"] = gdf.geometry.to_wkt()
building_csv = pd.DataFrame(
    gdf.drop(columns="geometry")
)
building_csv.to_csv(
    BUILDING_CSV_OUTPUT,
    index=False,
)

gdf.to_file(
    BUILDING_OUTPUT,
    layer="building_level_results",
    driver="GPKG",
)

print("All final outputs were written successfully.")

In [ ]:
# ---------------------------------------------------------------------
# Manuscript consistency checks
# ---------------------------------------------------------------------

EXPECTED_TABLE3 = {
    "Building": (0.0413, 0.00, 0.978),
    "100 m grid": (0.0329, 20.24, 0.920),
    "BGRI statistical subsections": (0.0309, 25.17, 0.903),
    "250 m grid": (0.0296, 28.13, 0.793),
    "500 m grid": (0.0267, 35.40, 0.599),
}

for _, row in table3.iterrows():
    expected_variance, expected_reduction, expected_moran = (
        EXPECTED_TABLE3[row["spatial_unit"]]
    )

    assert round(row["variance"], 4) == expected_variance
    assert (
        round(row["variance_reduction_percent"], 2)
        == expected_reduction
    )
    assert round(row["global_morans_i"], 3) == expected_moran

assert round(kmo_overall, 3) == 0.772
assert round(explained_variance_pc1 * 100, 2) == 50.46
assert round(equal_weight_correlation, 4) == 0.9804
assert round(population_index_correlation, 4) == -0.1057

print("All manuscript consistency checks passed.")

## Outputs

The notebook creates:

- `outputs/tables/pca_weights.csv`
- `outputs/tables/pca_diagnostics.csv`
- `outputs/tables/moran_knn_sensitivity.csv`
- `outputs/tables/lisa_cluster_counts.csv`
- `outputs/tables/table3_spatial_aggregation_knn16.csv`
- `outputs/tables/building_level_results.csv`
- `outputs/spatial/building_level_results.gpkg`
- `outputs/spatial/bgri_aggregated.gpkg`
- `outputs/spatial/grid_100m.gpkg`
- `outputs/spatial/grid_250m.gpkg`
- `outputs/spatial/grid_500m.gpkg`

In [ ]:
# ---------------------------------------------------------------------
# Population-flow verification
# ---------------------------------------------------------------------

accessibility_ids = set(accessibility["osm_id"].dropna())
population_ids = set(population["osm_id"].dropna())

population_total_redistributed = population["pop_64_mais"].sum()

population_in_accessibility_buildings = population.loc[
    population["osm_id"].isin(accessibility_ids),
    "pop_64_mais",
].sum()

population_in_excluded_buildings = population.loc[
    ~population["osm_id"].isin(accessibility_ids),
    "pop_64_mais",
].sum()

buildings_in_population_file = len(population)
buildings_in_accessibility_file = len(accessibility)

population_buildings_excluded = population.loc[
    ~population["osm_id"].isin(accessibility_ids)
].copy()

print("POPULATION FLOW")
print("-" * 50)
print(
    f"Buildings in population dataset: "
    f"{buildings_in_population_file:,}"
)
print(
    f"Buildings in final accessibility dataset: "
    f"{buildings_in_accessibility_file:,}"
)
print(
    f"Buildings excluded from the accessibility dataset: "
    f"{len(population_buildings_excluded):,}"
)
print()
print(
    f"Older residents redistributed to buildings: "
    f"{population_total_redistributed:,.0f}"
)
print(
    f"Older residents in buildings retained for analysis: "
    f"{population_in_accessibility_buildings:,.0f}"
)
print(
    f"Older residents in buildings excluded from analysis: "
    f"{population_in_excluded_buildings:,.0f}"
)

assert np.isclose(
    population_total_redistributed,
    population_in_accessibility_buildings
    + population_in_excluded_buildings,
)

print("\nPopulation accounting is internally consistent.")

In [ ]:
print(gdf["pop_64_mais"].sum())

print(
    gdf.loc[
        gdf["numero_servicos_proximos"] == 0,
        "pop_64_mais"
    ].sum()
)

print(
    gdf.loc[
        gdf["numero_servicos_proximos"] > 0,
        "pop_64_mais"
    ].sum()
)